# Bước 3 — Tạo embedding
Embedding biến mỗi chunk thành một vector số. Lần chạy đầu, model sẽ được tải từ Hugging Face.

In [ ]:
from pathlib import Path
import json
import numpy as np
from sentence_transformers import SentenceTransformer

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
STORAGE_DIR = PROJECT_DIR / 'storage'
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
chunks = json.loads((STORAGE_DIR / 'chunks.json').read_text(encoding='utf-8'))
texts = [item['content'] for item in chunks]
print('Số chunk:', len(texts))
print('Ví dụ:', texts[0][:300])

## Nạp model
Model chỉ dùng để biến văn bản thành vector; đây chưa phải LLM tạo câu trả lời.

In [ ]:
model = SentenceTransformer(MODEL_NAME)

## Thử embedding hai câu
Các vector đã chuẩn hóa nên tích vô hướng có thể dùng làm cosine similarity.

In [ ]:
examples = ['Nhân viên có bao nhiêu ngày phép?', 'Quy định nghỉ phép hằng năm']
example_vectors = model.encode(examples, normalize_embeddings=True)
print('Shape:', example_vectors.shape)
print('10 số đầu:', example_vectors[0][:10])
print('Độ tương đồng:', float(example_vectors[0] @ example_vectors[1]))

## Tạo embedding cho toàn bộ chunk

In [ ]:
embeddings = model.encode(texts, batch_size=16, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.asarray(embeddings, dtype='float32')
print('Kích thước ma trận:', embeddings.shape)

In [ ]:
output = STORAGE_DIR / 'embeddings.npy'
np.save(output, embeddings)
print('Đã lưu:', output)